In [0]:
from pyspark.sql.functions import col,lit,max,sum,current_timestamp
import uuid
from datetime import datetime

In [0]:
spark.sql("""
          create table if not exists novacart.bronze.ingestioncontrol(
              layer string,
              table string,
              ts_col string,
              pk_col string,
              last_successful_ts timestamp,
              last_successful_pk bigint,
              last_run_id string,
              rows_written bigint,
              row_status string,
              updated_at timestamp
          )
          """)

In [0]:
tables_config = {
    "orders":{"pk_col" : "order_id","ts_col" : "updated_at"},
    "payments":{"pk_col" : "payment_id","ts_col" : "processed_at"},
    "products":{"pk_col" : "product_id","ts_col" : "updated_at"}
}
bronze_run_id = str(uuid.uuid4())
print(bronze_run_id)

In [0]:
def get_last_successfil_watermark(table_name:str):
    cntrl = (
        spark.table("novacart.bronze.ingestioncontrol")
             .filter((col("layer") == "bronze") & (col("table") == table_name) & (col("row_status") == "Success"))
             .orderBy(col("updated_at").desc())
             .limit(1)
    )
    rows = cntrl.collect()
    if not rows:
        return None,None
    
    return rows[0].last_successful_ts,rows[0].last_successful_pk



In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):
    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col ,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written) if rows_written is not None else None,
            "Success",
            datetime.utcnow()
        )],
        schema=(
            "layer string, table string, ts_col string, pk_col string, last_successful_ts timestamp,last_successful_pk bigint,last_run_id string, rows_written bigint, row_status string,updated_at timestamp"
        )
    )
    control_df.createOrReplaceTempView("control_table")
    spark.sql("""
          merge into novacart.bronze.ingestioncontrol as t using control_table as s on t.`table` = s.`table` and t.`layer` = s.`layer`
          when matched then update set 
          t.ts_col = s.ts_col,
          t.pk_col = s.pk_col,
          t.last_successful_ts = s.last_successful_ts,
          t.last_successful_pk = s.last_successful_pk,
          t.last_run_id = s.last_run_id,
          t.rows_written = s.rows_written,
          t.row_status = s.row_status,
          t.updated_at = s.updated_at
          when not matched then insert *      
          """)


In [0]:
for table_name,cfg in tables_config.items():
    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]
    source_table = f"novacart.raw.{table_name}"
    target_table  = f"novacart.bronze.{table_name}"
    last_successful_ts,last_successful_pk = get_last_successfil_watermark(table_name)
    source_df = spark.table(source_table).withColumn(ts_col,col(ts_col).cast("timestamp"))
    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = (
            source_df.filter((col(ts_col) > lit(last_successful_ts)) | ((col(ts_col) == lit(last_successful_ts)) & (col(pk_col) > lit(last_successful_pk)))))
    
    rows_to_load = (
        rows_to_load.withColumn("bronze_ingested_at",current_timestamp())
                     .withColumn("bronze_run_id",lit(bronze_run_id))
                     .withColumn("bronze_source_table",lit(source_table))
    )
    rows_count = rows_to_load.count()
    if rows_count == 0:
        print(f"No new records found for {table_name}")
        upsert_bronze_control(table_name,ts_col,pk_col,last_successful_ts,last_successful_pk,rows_count,bronze_run_id)
        continue
        
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)
    max_ts = rows_to_load.agg(max(col(ts_col)).alias("max_ts")).collect()[0]["max_ts"]
    max_pk = (
            rows_to_load
            .filter(col(ts_col) == lit(max_ts))
            .agg(max(col(pk_col)).alias("max_pk"))
            .collect()[0]["max_pk"]
        )
    upsert_bronze_control(table_name,ts_col,pk_col,max_ts,max_pk,rows_count,bronze_run_id)
    
       


In [0]:
%sql
select * from novacart.bronze.ingestioncontrol;

In [0]:
%sql
select * from novacart.bronze.products;